# Week 63, Day 3: __getattr__

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Shadow:
    def __init__(self):
        self.real = "found normally"
    def __getattr__(self, name):
        return f"made up: {name}"

s = Shadow()
print(s.real)
print(s.anything)
print(hasattr(s, "whatever"), getattr(s, "x", "default"))
s.anything = "now real"
print(s.anything)

In [ ]:
import copy

class Record:
    def __init__(self, **fields):
        self._fields = fields

    def __getattr__(self, name):
        if name.startswith("_"):                  # never invent private or special names
            raise AttributeError(name)
        try:
            return self._fields[name]
        except KeyError:
            raise AttributeError(f"Record has no field {name!r}") from None

r = Record(name="Ada", email="ada@example.com")
print(r.name, hasattr(r, "phone"), getattr(r, "phone", "none"))
twin = copy.copy(r)                               # safe: _fields lookups during copying raise properly
print(twin.email)

In [ ]:
class Audited:
    def __init__(self, **values):
        object.__setattr__(self, "reads", [])
        for k, v in values.items():
            object.__setattr__(self, k, v)

    def __getattribute__(self, name):
        if not name.startswith("_") and name != "reads":
            object.__getattribute__(self, "reads").append(name)
        return object.__getattribute__(self, name)

a = Audited(x=1, y=2)
print(a.x + a.y + a.x)
print(a.reads)

In [ ]:
class Tracked:
    def __init__(self, **values):
        object.__setattr__(self, "changes", [])
        for k, v in values.items():
            setattr(self, k, v)

    def __setattr__(self, name, value):
        old = self.__dict__.get(name, "<unset>")
        self.changes.append((name, old, value))
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        self.changes.append((name, self.__dict__.get(name), "<deleted>"))
        object.__delattr__(self, name)

t = Tracked(colour="red")
t.colour = "blue"
del t.colour
print(t.changes)

In [ ]:
class Proxy:
    def __init__(self, target):
        object.__setattr__(self, "_target", target)
    def __getattr__(self, name):
        return getattr(self._target, name)

p = Proxy([3, 1, 2])
print(p.count(1), p.__len__())        # explicit attribute lookups are forwarded
try:
    len(p)                            # the special-method lookup isn't
except TypeError as e:
    print("TypeError:", e)

for special in ("__len__", "__getitem__", "__iter__"):
    setattr(Proxy, special, lambda self, *a, _n=special: getattr(self._target, _n)(*a))
print(len(p), p[0], sorted(p))

In [ ]:
import types

lazy = types.ModuleType("lazy")
loaded = []

def module_getattr(name):
    if name == "heavy":
        loaded.append(name)
        value = list(range(5))
        setattr(lazy, name, value)        # cache on the module
        return value
    raise AttributeError(name)

lazy.__getattr__ = module_getattr
print(loaded, lazy.heavy[-1], lazy.heavy[0], loaded)
print(hasattr(lazy, "other"))

> **Key idea.** __getattr__ is the fallback after normal lookup fails; it must raise AttributeError (so hasattr and getattr defaults work) and mustn't read attributes that might not exist yet, which happens during copying and unpickling. __getattribute__ runs for every read: use object.__getattribute__ inside it. __setattr__ and __delattr__ run for every assignment and deletion: hand on to object.__setattr__ / object.__delattr__, which keeps properties working. Special methods are looked up on the type and bypass all these hooks, so proxies must define them on the class. __dir__ advertises dynamic names, and a module-level __getattr__ gives lazy and deprecated module attributes.

### Exercise: A dict with attribute access

Write `AttrDict`, a subclass of `dict` where `d.key` reads `d["key"]`, `d.key = value` sets it and `del d.key` deletes it. Missing keys must raise **`AttributeError`** (so `hasattr` works), and names that start with `_` are never treated as keys. Normal dict methods (`d.items()`, `d.get`) keep working because they're found before `__getattr__`.

In [ ]:
class AttrDict(dict):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_attrdict():\n    \"attribute access to keys\"\n    d = AttrDict(host=\"localhost\", port=8080)\n    assert d.host == \"localhost\" and d.port == 8080\n    d.debug = True\n    assert d[\"debug\"] is True and \"debug\" not in vars(d)\n    del d.port\n    assert \"port\" not in d and not hasattr(d, \"port\") and getattr(d, \"port\", 1) == 1\n    try:\n        del d.missing\n        assert False\n    except AttributeError:\n        pass\n    assert sorted(d.items()) == [(\"debug\", True), (\"host\", \"localhost\")]\n    d[\"items\"] = \"shadowed\"\n    assert callable(d.items)\n    assert not hasattr(d, \"_private\")")

### Exercise: A counting proxy

Write `Proxy(target)`: reading any attribute it doesn't have forwards to the target and counts the name in `proxy_reads` (a dict from name to count); setting any attribute other than the proxy's own (`_target` and `proxy_reads`) sets it **on the target**; `len`, indexing and iteration work by forwarding (define `__len__`, `__getitem__` and `__iter__`). Set the proxy's own attributes with `object.__setattr__`.

In [ ]:
class Proxy:
    def __init__(self, target):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_proxy():\n    \"forwarding reads and writes\"\n    class Settings:\n        def __init__(self):\n            self.volume = 3\n    s = Settings()\n    p = Proxy(s)\n    assert p.volume == 3 and p.volume == 3 and p.proxy_reads == {\"volume\": 2}\n    p.volume = 9\n    assert s.volume == 9 and \"volume\" not in vars(p)\n    items = Proxy([4, 5, 6])\n    items.append(7)\n    assert len(items) == 4 and items[0] == 4 and list(items) == [4, 5, 6, 7]\n    assert items.proxy_reads == {\"append\": 1}\n    try:\n        p.missing\n        assert False\n    except AttributeError:\n        pass")

### Exercise: Remembering every change

Write a base class `History` whose subclasses record every attribute change: `__setattr__` appends `(name, old_value, new_value)` to the object's change list (old is `None` the first time) and then sets the attribute **through `object.__setattr__`**, so properties in subclasses still work. `changes()` returns a copy of the list and `undo()` reverts the latest change (deleting the attribute if its old value was the first-time `None`), without recording the undo itself.

In [ ]:
class History:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_history():\n    \"tracking with __setattr__\"\n    class Doc(History):\n        def __init__(self, title):\n            super().__init__()\n            self.title = title\n        @property\n        def words(self):\n            return self._words\n        @words.setter\n        def words(self, value):\n            object.__setattr__(self, \"_words\", max(0, value))\n        @words.deleter\n        def words(self):\n            object.__delattr__(self, \"_words\")\n    d = Doc(\"Draft\")\n    d.title = \"Final\"\n    d.words = -5\n    assert d.words == 0 and d.title == \"Final\"\n    assert d.changes() == [(\"title\", None, \"Draft\"), (\"title\", \"Draft\", \"Final\"), (\"words\", None, -5)]\n    d.undo()\n    assert len(d.changes()) == 2\n    d.undo()\n    assert d.title == \"Draft\" and d.changes() == [(\"title\", None, \"Draft\")]\n    d.undo()\n    assert not hasattr(d, \"title\") and d.changes() == []\n    d.changes().append(\"x\")\n    assert d.changes() == []")

### Exercise: Watching every read

Write a base class `Watched` that uses `__getattribute__` to count every read of a **public** attribute (one not starting with `_`), including method lookups and reads inside the object's own methods, in a dict available as `obj._reads`. Reads of private names aren't counted. `most_read()` returns the most-read name (ties: the alphabetically first), or `None`, and calling it isn't itself counted. Use `object.__getattribute__` to avoid recursion.

In [ ]:
class Watched:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_audit():\n    \"__getattribute__\"\n    class Counter(Watched):\n        def __init__(self):\n            super().__init__()\n            self.value = 0\n        def bump(self):\n            self.value += 1\n    c = Counter()\n    c.bump()\n    c.bump()\n    assert c._reads == {\"bump\": 2, \"value\": 2} and c.value == 2\n    assert c._reads[\"value\"] == 3\n    assert c.most_read() == \"value\"\n    assert \"most_read\" not in c._reads\n    assert Counter().most_read() is None\n    c.label = \"x\"\n    c.label\n    c.label\n    c.label\n    assert c.most_read() == \"label\" and c._reads[\"label\"] == 3")

### Exercise: Load on first use

Write `LazyNamespace(**loaders)`: each keyword is a function of no arguments. Reading `ns.name` for the first time calls that loader and **caches** the result in the instance's `__dict__`, so `__getattr__` isn't involved again. `loaded()` lists names loaded so far (sorted), unknown names raise `AttributeError`, and `dir(ns)` includes every loader name (define `__dir__`).

In [ ]:
class LazyNamespace:
    def __init__(self, **loaders):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lazy():\n    \"caching in __getattr__\"\n    calls = []\n    def big():\n        calls.append(\"big\")\n        return list(range(1000))\n    ns = LazyNamespace(big=big, small=lambda: 1)\n    assert ns.loaded() == [] and calls == []\n    assert ns.big[-1] == 999 and ns.big[0] == 0 and calls == [\"big\"]\n    assert ns.loaded() == [\"big\"] and \"big\" in vars(ns)\n    assert \"small\" in dir(ns) and \"big\" in dir(ns) and \"loaded\" in dir(ns)\n    assert ns.small == 1 and ns.loaded() == [\"big\", \"small\"]\n    assert not hasattr(ns, \"other\")")

### Bug hunt: The wrong kind of error

`__getattr__` lets a `KeyError` escape, and `getattr` with a default only catches `AttributeError`, so this crashes with `KeyError` instead of printing `default`. Raise `AttributeError` for missing settings so it prints `default`.

In [ ]:
class Config:
    def __init__(self, values):
        self._values = values
    def __getattr__(self, name):
        return self._values[name]

cfg = Config({"mode": "fast"})
print(getattr(cfg, "colour", "default"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"AttributeError\"\n    expect_output(\"default\")")

### Bug hunt: A wrapper len can't see

The wrapper forwards attributes with `__getattr__`, but `len()` looks for `__len__` on the class directly, so this crashes with `TypeError`. Give the class its own `__len__` that forwards to the wrapped list so it prints `3`.

In [ ]:
class Wrapper:
    def __init__(self, items):
        self._items = items
    def __getattr__(self, name):
        return getattr(self._items, name)

w = Wrapper([1, 2, 3])
print(len(w))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"special methods\"\n    expect_output(\"3\")")

### Bug hunt: A setter that calls itself

`__setattr__` stores the value with `setattr(self, name, ...)`, which calls `__setattr__` again, and again, until `RecursionError`. Hand the work on to `object.__setattr__` so this prints `ADA`.

In [ ]:
class Shouting:
    def __setattr__(self, name, value):
        setattr(self, name, value.upper())

s = Shouting()
s.name = "ada"
print(s.name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"object.__setattr__\"\n    expect_output(\"ADA\")")

### Puzzle: A safe fallback

Arrange the lines of a `__getattr__` that reads from a dict and raises the right error. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: __getattr__ is called for attributes that exist
q = ___   # True or False: __getattribute__ is called for attributes that exist
r = ___   # the error __getattr__ should raise for a missing name, as a string
s = ___   # True or False: len(obj) goes through __getattr__ when __len__ is missing from the class
t = ___   # what to call inside __setattr__ to really store the value: "object.__setattr__" or "setattr"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q is True and r == \"AttributeError\", \"p, q and r\"\n    assert s is False and t == \"object.__setattr__\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: settings from the environment

Programs often read configuration from environment variables like `DB_HOST` and `DB_PORT`. Write `Env(mapping, prefix="")` giving attribute access to them, grouped by underscores:

- `env.db.host` reads the key `"DB_HOST"`: an attribute name is upper-cased and joined to the prefix with `_`. If that exact key exists, its value is returned, converted to `int` if it's all digits, to `True`/`False` if it's `"true"`/`"false"` (any case), and left as a string otherwise. If no exact key exists but some key **starts with** it plus `_`, a new `Env` for that prefix is returned. Otherwise `AttributeError`.
- Setting or deleting any attribute raises `AttributeError("Env is read-only")`.
- `dir(env)` lists the next-level names, lower-cased and sorted (for the root: the first part of every key).
- `env.as_dict()` returns every key under the prefix, with the prefix and its underscore removed, lower-cased, and values converted.
- `repr(env)` is `Env('DB')` (or `Env('')` for the root), and two `Env`s are equal if they have the same mapping and prefix.

Names starting with `_` are never looked up as keys.

In [ ]:
class Env:
    def __init__(self, mapping, prefix=""):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_env():\n    \"attribute access to environment variables\"\n    m = {\"DB_HOST\": \"db.local\", \"DB_PORT\": \"5432\", \"DB_POOL_SIZE\": \"10\", \"DEBUG\": \"True\", \"APP_NAME\": \"atlas\", \"LEVEL\": \"info\"}\n    env = Env(m)\n    assert env.db.host == \"db.local\" and env.db.port == 5432 and env.db.pool.size == 10\n    assert env.debug is True and env.level == \"info\" and env.app.name == \"atlas\"\n    assert isinstance(env.db, Env) and repr(env.db) == \"Env('DB')\" and repr(env) == \"Env('')\"\n    assert env.db == Env(m, \"DB\") and env.db != env.app\n    for action in (lambda: setattr(env, \"debug\", False), lambda: delattr(env, \"db\")):\n        try:\n            action()\n            assert False\n        except AttributeError as e:\n            assert str(e) == \"Env is read-only\"\n    assert not hasattr(env, \"missing\") and not hasattr(env.db, \"user\") and not hasattr(env, \"_secret\")\n    assert getattr(env.db, \"user\", \"postgres\") == \"postgres\"\n\ndef test_listing():\n    \"dir and as_dict\"\n    m = {\"DB_HOST\": \"h\", \"DB_PORT\": \"1\", \"DB_POOL_SIZE\": \"10\", \"DEBUG\": \"false\", \"APP_NAME\": \"atlas\"}\n    env = Env(m)\n    assert [n for n in dir(env) if not n.startswith(\"_\") and n not in (\"as_dict\",)] == [\"app\", \"db\", \"debug\"]\n    assert [n for n in dir(env.db) if not n.startswith(\"_\") and n != \"as_dict\"] == [\"host\", \"pool\", \"port\"]\n    assert env.db.as_dict() == {\"host\": \"h\", \"port\": 1, \"pool_size\": 10}\n    assert env.as_dict()[\"debug\"] is False and len(env.as_dict()) == 5\n    assert env.db.pool.as_dict() == {\"size\": 10}")

## Warm-up

`__getattr__` only runs when normal lookup fails, so once `anything` exists on the instance it's never called for that name. And because it never raises, `hasattr` says every attribute exists and `getattr`'s default is never used.

## __getattr__ in depth

`__getattr__(self, name)` is the **fallback**: Python calls it after the whole lookup order from yesterday has failed (or a property's getter raised `AttributeError`). Whatever it returns is the value; to say "no such attribute" it must **raise `AttributeError`**. That matters, because `hasattr(obj, name)` and `getattr(obj, name, default)` work by catching `AttributeError` and nothing else: raise a `KeyError` and they break.

Three patterns:

- **Forwarding** (Week 31): `return getattr(self._inner, name)`.
- **Computing**: build the attribute from data, for example `record.email` from `record._fields["email"]`.
- **Caching**: compute once, then `setattr(self, name, value)` (or write to `self.__dict__`), so the next read finds it normally and `__getattr__` isn't called again.

The classic trap: `__getattr__` reads `self._fields`, but `_fields` doesn't exist yet. That happens before `__init__` has run, which is exactly what `copy.copy`, `pickle` and similar tools do (they make the object with `__new__` and then look for hooks such as `__setstate__`). The lookup of `self._fields` fails, calls `__getattr__("_fields")`, which reads `self._fields`... and you get `RecursionError`. Read your own state with `self.__dict__` or `object.__getattribute__`, or refuse private names first.

## __getattribute__

`__getattribute__(self, name)` runs for **every** attribute read, before anything else: it **is** the lookup order (that's `object.__getattribute__`, the machinery from yesterday). Overriding it lets you see or change every access, including `self.anything` inside your own methods.

The rule: get real attributes by calling **`object.__getattribute__(self, name)`** (or `super().__getattribute__(name)`). Writing `self.__dict__` or `self.x` inside it calls `__getattribute__` again, and recurses forever. If `__getattribute__` raises `AttributeError` and the class also defines `__getattr__`, Python then calls `__getattr__`.

Because it runs on every access, it slows every attribute read and is rarely the right tool. It's used for things like auditing, access control and transparent proxies; prefer `__getattr__` or descriptors when they'll do.

## __setattr__ and __delattr__

`__setattr__(self, name, value)` runs for **every** assignment `obj.name = value`, including the ones in `__init__`, and `__delattr__(self, name)` for every `del obj.name`. Uses: validation, read-only objects (Week 62), change tracking, redirecting attributes into a dict or a remote store.

Do the real work by handing on to **`object.__setattr__(self, name, value)`** (or `super().__setattr__(name, value)`), which keeps descriptors and properties working. Writing `self.name = value` inside `__setattr__` calls it again, forever. Writing straight into `self.__dict__[name]` works but **skips data descriptors** such as properties and slots, which is usually a bug.

A common need is setting up internal state in `__init__` without triggering your own checks: use `object.__setattr__` for those.

## What the hooks don't see

**Special methods are looked up on the type, directly** (Week 31): `len(obj)`, `obj + 1`, `obj[0]`, `str(obj)`, `with obj:` and `for x in obj` find `__len__`, `__add__` and so on on `type(obj)`, without going through `__getattr__` or `__getattribute__`. So a forwarding proxy around a list supports `proxy.append(1)` but **not** `len(proxy)` or `proxy[0]`.

The fix is to define the special methods you need explicitly on the proxy class, each forwarding by hand, or generate them in a loop after the class is created (`setattr(Proxy, "__len__", ...)` on the class, which does work). Some libraries do exactly that for every special method.

The other gap: **`__getattr__` doesn't see attributes that exist**. If the class or an ancestor has a `name` method, a forwarding `__getattr__` never forwards `name`.

## __dir__ and module __getattr__

Attributes made up by `__getattr__` are invisible to `dir()`, tab completion and documentation tools. Define **`__dir__(self)`** to return the names you support (often `sorted(set(super().__dir__()) | your_names)`).

**Modules can have `__getattr__` too**: define a function `__getattr__(name)` at the top level of a module, and `module.missing` calls it (PEP 562). Uses: **lazy imports** (load a heavy submodule the first time it's touched), **deprecation warnings** for renamed names, and computed constants. A module-level `__dir__()` function works the same way for `dir(module)`.

```
# shapes.py
import importlib

_LAZY = {"plotting": "shapes._plotting"}

def __getattr__(name):
    if name in _LAZY:
        module = importlib.import_module(_LAZY[name])
        globals()[name] = module          # cache: next time it's found normally
        return module
    raise AttributeError(f"module 'shapes' has no attribute {name!r}")

def __dir__():
    return sorted(list(globals()) + list(_LAZY))
```

## Practice

## Quiz

## Daily challenge

## Before you go

Run `dir()` on a module you use often and look for names you didn't expect; then look up PEP 562 to see the examples it gives for module `__getattr__`. Tomorrow: metaclasses, the classes that make classes.